# Analisis Text Mining dan Coding Juz 1

Cakupan: Al-Fātiḥah 1–7 dan Al-Baqarah 1–141. Analisis memakai terjemahan Indonesia; teks sumber dan rujukan ayat tetap disimpan.

Hasil akhir notebook: coding lengkap untuk kandidat “orang-orang yang beriman” pada 9 ayat, dengan tema dan interpretasi terbatas pada bukti tersebut. Ini bukan kesimpulan untuk seluruh Juz 1.

## 1. Memuat dan memeriksa dataset

Baca CSV, lalu periksa kolom, nilai kosong, dan keunikan ayat.

In [ ]:
from pathlib import Path
import re
from collections import Counter

import pandas as pd
from IPython.display import display

relative_csv = Path("data") / "raw" / "quran_dataset.csv"
candidate_paths = [Path.cwd() / relative_csv, Path.cwd().parent / relative_csv]
csv_path = next((path for path in candidate_paths if path.is_file()), None)

if csv_path is None:
    checked_paths = ", ".join(str(path) for path in candidate_paths)
    raise FileNotFoundError(f"Dataset tidak ditemukan. Lokasi yang diperiksa: {checked_paths}")

data = pd.read_csv(csv_path, encoding="utf-8-sig")
required_columns = [
    "nomor_surat",
    "nama_surat",
    "nomor_ayat",
    "teks_arab",
    "teks_terjemahan",
]
missing_columns = [column for column in required_columns if column not in data.columns]
if missing_columns:
    raise ValueError(f"Kolom wajib tidak ditemukan: {missing_columns}")

missing_values = data[required_columns].isna().sum()
duplicate_verses = data.duplicated(subset=["nomor_surat", "nomor_ayat"]).sum()
if missing_values.any():
    raise ValueError(f"Nilai kosong ditemukan pada kolom wajib:\n{missing_values[missing_values > 0]}")
if duplicate_verses:
    raise ValueError(f"Ditemukan {duplicate_verses} rujukan surah-ayat duplikat.")

print(f"Dataset: {csv_path}")
print(f"Jumlah ayat: {len(data):,}")
print(f"Jumlah surah: {data['nomor_surat'].nunique()}")
print("Nilai kosong pada kolom wajib: 0")
print("Rujukan surah-ayat duplikat: 0")
display(data.head(3))

CSV terbaca dengan lima kolom yang diperlukan. Pemeriksaan memastikan tidak ada nilai kosong atau rujukan ayat ganda pada kolom wajib.

## 2. Menyaring Juz 1

Gunakan nomor surah dan ayat agar batas Juz 1 tidak bergantung pada penulisan nama surah.

In [ ]:
surah_1 = data.loc[
    (data["nomor_surat"] == 1) & data["nomor_ayat"].between(1, 7)
]
surah_2 = data.loc[
    (data["nomor_surat"] == 2) & data["nomor_ayat"].between(1, 141)
]
ayat_juz1 = pd.concat([surah_1, surah_2], ignore_index=True)

expected_refs = {(1, ayah) for ayah in range(1, 8)}
expected_refs.update((2, ayah) for ayah in range(1, 142))
actual_refs = set(zip(ayat_juz1["nomor_surat"], ayat_juz1["nomor_ayat"]))

if len(ayat_juz1) != 148:
    raise ValueError(f"Jumlah ayat Juz 1 seharusnya 148, ditemukan {len(ayat_juz1)}.")
if actual_refs != expected_refs:
    missing_refs = sorted(expected_refs - actual_refs)
    unexpected_refs = sorted(actual_refs - expected_refs)
    raise ValueError(f"Rujukan Juz 1 tidak sesuai. Hilang: {missing_refs}; di luar cakupan: {unexpected_refs}")
if ayat_juz1.duplicated(subset=["nomor_surat", "nomor_ayat"]).any():
    raise ValueError("Hasil filter mengandung rujukan surah-ayat duplikat.")

print(f"Total ayat Juz 1: {len(ayat_juz1)}")
print(f"Al-Fātiḥah: {len(surah_1)} ayat (1–7)")
print(f"Al-Baqarah: {len(surah_2)} ayat (1–141)")
display(ayat_juz1[["nomor_surat", "nama_surat", "nomor_ayat", "teks_terjemahan"]].head(3))
display(ayat_juz1[["nomor_surat", "nama_surat", "nomor_ayat", "teks_terjemahan"]].tail(3))

Hasil filter berisi 148 ayat: 7 ayat Al-Fātiḥah dan 141 ayat Al-Baqarah. Set rujukannya divalidasi terhadap seluruh nomor ayat yang diharapkan.

## 3. Preprocessing ringan

Buat salinan pencarian dengan spasi/baris baru dirapikan dan huruf diseragamkan. Teks asli tidak diubah; tanda baca, urutan kata, dan negasi tetap ada.

In [ ]:
jumlah_ayat_sebelum = len(ayat_juz1)
teks_asli_sebelum = ayat_juz1["teks_terjemahan"].copy()

ayat_analisis = ayat_juz1.copy()
ayat_analisis["teks_pencarian"] = (
    ayat_analisis["teks_terjemahan"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .str.lower()
)

pola_negasi = {
    "tidak": r"\btidak\b",
    "jangan": r"\bjangan\w*\b",
}
jumlah_negasi_asli = {
    kata: sum(len(re.findall(pola, teks.lower())) for teks in teks_asli_sebelum)
    for kata, pola in pola_negasi.items()
}
jumlah_negasi_bersih = {
    kata: sum(len(re.findall(pola, teks)) for teks in ayat_analisis["teks_pencarian"])
    for kata, pola in pola_negasi.items()
}

if len(ayat_analisis) != jumlah_ayat_sebelum:
    raise ValueError("Jumlah ayat berubah setelah preprocessing.")
if not ayat_analisis["teks_terjemahan"].equals(teks_asli_sebelum):
    raise ValueError("Kolom teks sumber berubah setelah preprocessing.")
if jumlah_negasi_asli != jumlah_negasi_bersih:
    raise ValueError(f"Jumlah negasi berubah: sebelum {jumlah_negasi_asli}; sesudah {jumlah_negasi_bersih}")
if ayat_analisis["teks_pencarian"].str.contains(r"\s{2,}", regex=True).any():
    raise ValueError("Masih ditemukan spasi berulang pada teks pencarian.")

print(f"Jumlah ayat dipertahankan: {len(ayat_analisis)}")
print(f"Jumlah negasi sebelum: {jumlah_negasi_asli}")
print(f"Jumlah negasi sesudah: {jumlah_negasi_bersih}")
display(
    ayat_analisis[["nomor_surat", "nomor_ayat", "teks_terjemahan", "teks_pencarian"]].head(3)
)

Preprocessing hanya membuat kolom pencarian huruf kecil dengan whitespace dirapikan. Teks sumber dan 148 rujukan tetap utuh; “tidak” tetap 76 dan “jangan/janganlah” tetap 12 kemunculan.

## 4. Kandidat keyword

Hitung token pada salinan pencarian. Daftar kata umum hanya dipakai untuk tabel kandidat; kata tersebut tidak dihapus dari teks atau konteks ayat.

In [ ]:
kata_umum = {
    "yang", "dan", "di", "ke", "dari", "pada", "dengan", "untuk", "ini", "itu",
    "mereka", "kami", "kamu", "engkau", "dia", "ia", "akan", "telah", "sudah",
    "ada", "oleh", "bagi", "antara", "dalam", "atas", "karena", "maka", "lalu",
    "apabila", "ketika", "sebagai", "lagi", "pun", "suatu", "apa", "sesungguhnya",
    "kepada", "adalah", "al",
}
if kata_umum.intersection({"tidak", "jangan", "janganlah"}):
    raise ValueError("Kata negasi tidak boleh dimasukkan ke daftar kata umum.")

token_per_ayat = [
    re.findall(r"\b[a-z]+(?:['-][a-z]+)*\b", teks)
    for teks in ayat_analisis["teks_pencarian"]
]
frekuensi_token = Counter(token for token_list in token_per_ayat for token in token_list)
token_terpilih = [
    (token, frekuensi)
    for token, frekuensi in frekuensi_token.most_common()
    if token not in kata_umum
][:15]

baris_kandidat = []
for token, frekuensi in token_terpilih:
    ayat_yang_memuat = [
        row
        for row in ayat_analisis.itertuples(index=False)
        if re.search(rf"\b{re.escape(token)}\b", row.teks_pencarian)
    ]
    baris_kandidat.append(
        {
            "jenis": "keyword",
            "kandidat": token,
            "frekuensi_kata": frekuensi,
            "jumlah_ayat": len(ayat_yang_memuat),
            "rujukan_ayat": ", ".join(
                f"{row.nama_surat} {row.nomor_surat}:{row.nomor_ayat}" for row in ayat_yang_memuat
            ),
            "contoh_kutipan": " | ".join(
                f"{row.nama_surat} {row.nomor_surat}:{row.nomor_ayat} — {row.teks_terjemahan.strip()}"
                for row in ayat_yang_memuat[:2]
            ),
        }
    )

kandidat_keyword_awal = pd.DataFrame(baris_kandidat)
print(f"Jumlah token unik: {len(frekuensi_token):,}")
print("Negasi dipertahankan dalam teks dan tidak masuk daftar kata umum.")
display(kandidat_keyword_awal)

Tabel menampilkan 15 keyword awal dengan frekuensi, jumlah ayat, rujukan, dan maksimal dua kutipan konteks. Frekuensi bukan ukuran kepentingan; pilih kandidat sesuai tujuan eksplorasi.

## 5. Kandidat phrase dan konteks ayat

Fokus sementara: eksplorasi kandidat kata/frasa di seluruh Juz 1. Phrase dipilih manual agar variasi konteks, tindakan, dan negasi ikut terlihat; ini bukan tema final.

In [ ]:
phrase_eksploratif = [
    "orang-orang yang beriman",
    "beriman kepada Allah",
    "berbuat baik",
    "janganlah kamu",
    "mengikuti petunjuk",
    "orang-orang yang kafir",
    "hari kemudian",
    "beriman kepada yang ghaib",
    "mendirikan shalat",
    "menafkahkan sebahagian rezeki",
    "tidak ada keraguan",
]

batas_contoh_kutipan = 2
baris_phrase = []
for phrase in phrase_eksploratif:
    pola = rf"(?<!\w){re.escape(phrase.lower())}(?!\w)"
    ayat_yang_memuat = ayat_analisis.loc[
        ayat_analisis["teks_pencarian"].str.contains(pola, regex=True)
    ]
    jumlah_kemunculan = sum(
        len(re.findall(pola, teks)) for teks in ayat_yang_memuat["teks_pencarian"]
    )
    baris_phrase.append(
        {
            "jenis": "phrase",
            "phrase": phrase,
            "frekuensi": jumlah_kemunculan,
            "jumlah_ayat": len(ayat_yang_memuat),
            "rujukan_ayat": "; ".join(
                f"{row.nama_surat} {row.nomor_surat}:{row.nomor_ayat}" for row in ayat_yang_memuat.itertuples()
            ),
            "contoh_kutipan": " | ".join(
                f"{row.nama_surat} {row.nomor_surat}:{row.nomor_ayat} — {row.teks_terjemahan.strip()}"
                for row in ayat_yang_memuat.head(batas_contoh_kutipan).itertuples()
            ),
        }
    )

kandidat_phrase_awal = pd.DataFrame(baris_phrase)
display(kandidat_phrase_awal)

baris_konteks_phrase = []
for phrase in phrase_eksploratif:
    pola = rf"(?<!\w){re.escape(phrase.lower())}(?!\w)"
    ayat_yang_memuat = ayat_analisis.loc[
        ayat_analisis["teks_pencarian"].str.contains(pola, regex=True)
    ]
    for row in ayat_yang_memuat.itertuples():
        baris_konteks_phrase.append(
            {
                "phrase": phrase,
                "rujukan_ayat": f"{row.nama_surat} {row.nomor_surat}:{row.nomor_ayat}",
                "teks_terjemahan_lengkap": row.teks_terjemahan.strip(),
            }
        )

konteks_lengkap_phrase = pd.DataFrame(baris_konteks_phrase)
print(f"Semua konteks untuk phrase yang cocok: {len(konteks_lengkap_phrase)} ayat")
with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
    display(konteks_lengkap_phrase)

Tabel ringkasan memuat semua rujukan, tetapi hanya dua kutipan contoh. Tabel **konteks lengkap** di atas menampilkan seluruh ayat yang cocok, satu ayat per baris, dengan terjemahan utuh agar bisa disalin ke prompt. Frekuensi nol berarti ejaan persis tidak ditemukan.

## 6. Meminta AI menilai makna kandidat

Pilih satu keyword atau phrase dan salin prompt lengkap yang terbentuk ke AI web pilihanmu. Jangan hanya menyalin tabel ringkasan karena sebagian kutipan bisa dipotong.

Prompt memuat seluruh ayat yang cocok, rujukan, dan frekuensi kandidat. AI hanya memberi usulan; keputusan coding tetap milik peneliti.

In [ ]:
import pandas as pd

jenis_untuk_ai = "phrase"
kandidat_untuk_ai = "orang-orang yang beriman"

if jenis_untuk_ai not in {"keyword", "phrase"}:
    raise ValueError("jenis_untuk_ai harus 'keyword' atau 'phrase'.")
if not kandidat_untuk_ai.strip():
    raise ValueError("kandidat_untuk_ai tidak boleh kosong.")

kandidat_normal = kandidat_untuk_ai.strip().lower()
pola_kandidat = rf"(?<!\w){re.escape(kandidat_normal)}(?!\w)"

ayat_kandidat_ai = ayat_analisis.loc[
    ayat_analisis["teks_pencarian"].str.contains(pola_kandidat, regex=True)
]
if ayat_kandidat_ai.empty:
    raise ValueError(f"Kandidat tidak ditemukan secara literal: {kandidat_normal}")

frekuensi_kandidat_ai = sum(
    len(re.findall(pola_kandidat, teks))
    for teks in ayat_kandidat_ai["teks_pencarian"]
)
konteks_ai = [
    f"- {row.nama_surat} {row.nomor_surat}:{row.nomor_ayat} — {row.teks_terjemahan.strip()}"
    for row in ayat_kandidat_ai.itertuples()
]
rujukan_ai = [
    f"{row.nama_surat} {row.nomor_surat}:{row.nomor_ayat}"
    for row in ayat_kandidat_ai.itertuples()
]
prompt_ai = f"""Saya menganalisis terjemahan bahasa Indonesia Al-Qur’an pada seluruh Juz 1.
Tujuan eksploratif: menilai makna kandidat berdasarkan konteks yang diberikan, bukan frekuensi saja.

Jenis kandidat: {jenis_untuk_ai}
Kandidat: {kandidat_normal}
Frekuensi literal dalam ayat Juz 1: {frekuensi_kandidat_ai}
Jumlah ayat yang memuat kandidat: {len(ayat_kandidat_ai)}

Jelaskan makna dalam konteks kutipan, tunjukkan bukti tekstual, pertahankan rujukan persis, dan nyatakan keterbatasan. Jangan menambahkan klaim atau tafsir yang tidak didukung kutipan. Frekuensi bukan ukuran kepentingan makna.
Jawab ringkas dengan: kandidat | makna kontekstual | bukti dan rujukan | keterbatasan.

Kutipan lengkap:
{chr(10).join(konteks_ai)}"""

print("=== SALIN SELURUH PROMPT DI BAWAH INI KE AI WEB ===\n")
print(prompt_ai)

catatan_ai = pd.DataFrame([{
    "jenis_kandidat": jenis_untuk_ai,
    "kandidat": kandidat_normal,
    "frekuensi": frekuensi_kandidat_ai,
    "jumlah_ayat": len(ayat_kandidat_ai),
    "rujukan": "; ".join(rujukan_ai),
}])
display(catatan_ai)

## 7. Respons dan evaluasi AI

Jawaban ChatGPT yang diberikan pengguna disimpan utuh di bawah sebagai sumber analisis. Penilaian kecocokan bukti dan keterbatasannya dirangkum pada bagian Evaluasi AI setelah tahap coding. Notebook tidak membuat klaim bahwa ada panggilan atau jawaban AI eksternal baru.

### Respons AI web: “orang-orang yang beriman”

Platform: ChatGPT. Tanggal: 22 Oktober 2026.

Kandidat frasa “orang-orang yang beriman” muncul sebanyak 9 kali dalam dataset terjemahan Al-Qur'an Juz 1, berdasarkan hitungan literal yang diberikan. Namun, konteks penggunaannya beragam: ada yang berkaitan dengan klaim keimanan, interaksi antarkelompok, penerimaan perumpamaan, amal saleh, petunjuk Al-Qur'an, dan perintah kepada orang-orang beriman.

Analisis berikut membedakan fungsi frasa pada setiap ayat tanpa menjadikan frekuensi sebagai ukuran kepentingan makna atau menetapkan tema final.

### Al-Baqara 2:8

**Konteks:** Frasa muncul dalam pernyataan yang membedakan pengakuan keimanan dari keadaan sebenarnya.

**Makna menurut AI:** Frasa merujuk pada orang-orang yang benar-benar memiliki keadaan beriman, sebagai pembanding bagi pihak yang mengaku beriman tetapi dinyatakan bukan bagian dari kelompok tersebut.

**Bukti tekstual:** “Kami beriman kepada Allah dan Hari kemudian,” dan “sesungguhnya bukan orang-orang yang beriman.”

**Batasan:** Kutipan menunjukkan perbedaan antara pengakuan dan keadaan keimanan. Namun, kriteria terperinci untuk menentukan siapa yang benar-benar beriman tidak dijelaskan dalam kutipan.

### Al-Baqara 2:9

**Konteks:** Orang-orang beriman menjadi pihak yang hendak ditipu oleh kelompok yang dibicarakan.

**Makna menurut AI:** Frasa menunjuk pada pihak yang menjadi sasaran upaya penipuan. Mereka ditempatkan dalam hubungan sosial yang berlawanan dengan kelompok yang berusaha menipu.

**Bukti tekstual:** “Mereka hendak menipu Allah dan orang-orang yang beriman.”

**Batasan:** Kutipan tidak menjelaskan secara terperinci siapa saja yang termasuk dalam kelompok orang-orang beriman atau bagaimana upaya penipuan tersebut berlangsung.

### Al-Baqara 2:14

**Konteks:** Frasa muncul dalam konteks perjumpaan dan percakapan antara dua kelompok.

**Makna menurut AI:** Orang-orang beriman merupakan kelompok yang dihadapi oleh pihak yang mengaku telah beriman, sementara pengakuan tersebut berbeda dari pernyataan mereka ketika kembali kepada kelompoknya sendiri.

**Bukti tekstual:** “Bila mereka berjumpa dengan orang-orang yang beriman, mereka mengatakan: ‘Kami telah beriman’”; kemudian, “kami hanyalah berolok-olok.”

**Batasan:** Kutipan menunjukkan perbedaan pernyataan berdasarkan lawan bicara. Identitas lengkap kedua kelompok dan latar perjumpaannya tidak dapat ditentukan hanya dari kutipan ini.

### Al-Baqara 2:26

**Konteks:** Orang-orang beriman dibandingkan dengan orang-orang kafir dalam menanggapi perumpamaan yang dibuat Allah.

**Makna menurut AI:** Frasa merujuk pada kelompok yang menerima perumpamaan tersebut sebagai sesuatu yang benar dari Tuhan mereka. Dalam konteks ini, ciri yang ditonjolkan adalah keyakinan terhadap kebenaran perumpamaan.

**Bukti tekstual:** “Adapun orang-orang yang beriman, maka mereka yakin bahwa perumpamaan itu benar dari Tuhan mereka.”

**Batasan:** Kutipan memperlihatkan perbedaan tanggapan terhadap perumpamaan. Namun, kutipan tidak menjelaskan seluruh alasan di balik perbedaan tanggapan tersebut.

### Al-Baqara 2:76

**Konteks:** Orang-orang beriman menjadi pihak yang ditemui oleh kelompok lain dalam situasi percakapan mengenai pengakuan keimanan dan informasi yang disampaikan.

**Makna menurut AI:** Frasa menunjuk pada kelompok yang menerima pernyataan “Kamipun telah beriman” dari pihak yang sedang dibicarakan. Mereka juga disebut dalam percakapan internal kelompok tersebut sebagai orang yang mungkin menggunakan informasi tertentu untuk mengalahkan hujah mereka.

**Bukti tekstual:** “Apabila mereka berjumpa dengan orang-orang yang beriman, mereka berkata: ‘Kamipun telah beriman’”; serta “supaya dengan demikian mereka dapat mengalahkan hujjahmu.”

**Batasan:** Kutipan memperlihatkan hubungan komunikasi dan kekhawatiran mengenai penggunaan informasi. Tidak cukup bukti untuk menyimpulkan seluruh hubungan sosial atau maksud setiap pihak di luar pernyataan yang dikutip.

### Al-Baqara 2:82

**Konteks:** Frasa digunakan bersama keterangan mengenai amal saleh dan konsekuensi yang dinyatakan bagi kelompok tersebut.

**Makna menurut AI:** Orang-orang beriman digambarkan sebagai bagian dari kelompok yang disebut bersama pelaku amal saleh. Dalam kutipan, kelompok tersebut dikaitkan dengan surga dan keberadaan yang kekal di dalamnya.

**Bukti tekstual:** “Orang-orang yang beriman serta beramal saleh, mereka itu penghuni surga; mereka kekal di dalamnya.”

**Batasan:** Kutipan menghubungkan keimanan dan amal saleh dengan konsekuensi yang disebutkan. Namun, kutipan ini sendiri tidak merinci bentuk amal saleh ataupun kriteria keimanan.

### Al-Baqara 2:91

**Konteks:** Frasa muncul dalam pertanyaan retoris mengenai konsistensi pengakuan keimanan ketika kelompok yang dibicarakan diminta beriman kepada Al-Qur’an.

**Makna menurut AI:** Frasa merujuk pada orang yang mengaku beriman, tetapi pengakuan tersebut dipertanyakan berdasarkan tindakan yang disebutkan dalam kutipan. Fokusnya adalah kesesuaian antara klaim keimanan dan tindakan terhadap nabi-nabi Allah.

**Bukti tekstual:** “Mengapa kamu dahulu membunuh nabi-nabi Allah jika benar kamu orang-orang yang beriman?”

**Batasan:** Kutipan menempatkan tindakan masa lalu sebagai dasar pertanyaan terhadap pengakuan keimanan. Namun, tidak cukup untuk menentukan identitas setiap orang yang dimaksud atau konteks historis lengkap peristiwa tersebut.

### Al-Baqara 2:97

**Konteks:** Frasa muncul pada akhir pernyataan mengenai Al-Qur’an yang diturunkan dengan izin Allah, membenarkan kitab-kitab sebelumnya, serta menjadi petunjuk dan berita gembira.

**Makna menurut AI:** Orang-orang beriman digambarkan sebagai pihak yang menjadi sasaran berita gembira yang disebutkan dalam kutipan. Frasa ini berfungsi mengidentifikasi penerima berita gembira tersebut.

**Bukti tekstual:** “Menjadi petunjuk serta berita gembira bagi orang-orang yang beriman.”

**Batasan:** Kutipan menyebutkan penerima berita gembira, tetapi tidak menjelaskan secara terperinci bentuk berita gembira atau karakteristik seluruh penerimanya.

### Al-Baqara 2:104

**Konteks:** Frasa digunakan sebagai sapaan langsung sebelum penyampaian larangan dan arahan mengenai pilihan kata.

**Makna menurut AI:** Orang-orang beriman diposisikan sebagai pihak yang dituju oleh perintah. Dalam konteks ini, frasa berfungsi sebagai penanda penerima instruksi, bukan sebagai bagian dari uraian tentang perbedaan kelompok atau konsekuensi keimanan.

**Bukti tekstual:** “Hai orang-orang yang beriman, janganlah kamu katakan (kepada Muhammad): ‘Raa'ina’, tetapi katakanlah: ‘Unzhurna’, dan ‘dengarlah’.”

**Batasan:** Kutipan menunjukkan adanya larangan dan arahan penggunaan kata. Alasan terperinci di balik pilihan kata tersebut tidak sepenuhnya dijelaskan dalam bagian kutipan yang diberikan.


### Membandingkan jawaban dengan AI lain

Jalankan prompt pada cell berikut, lalu salin prompt yang tampil ke AI pembanding. Minta jawaban dalam tabel Markdown dengan kolom yang sama. Tempel **seluruh tabel jawaban AI** ke variabel `LLM_AI_PEMBANDING` pada cell ISI MANUAL di bawah, persis seperti pola `LLM_STAGE = """..."""` pada notebook contoh. Isi nama model, penilaian, dan catatan Anda. Hasil akan dipasangkan dengan jawaban ChatGPT awal berdasarkan rujukan ayat.

Cell berikutnya berisi fungsi untuk membaca tabel Markdown hasil AI yang ditempel pada cell manual.

In [ ]:
def parse_ai_markdown_table(response, required_columns, stage_name):
    """Parse one Markdown response table and validate its required headers."""
    if not response.strip():
        return pd.DataFrame(columns=required_columns)

    lines = [line.strip() for line in response.splitlines() if line.strip()]

    def split_row(line):
        return [part.strip().strip("`") for part in line.strip().strip("|").split("|")]

    header_index = None
    headers = None
    for index, line in enumerate(lines[:-1]):
        if "|" not in line or "|" not in lines[index + 1]:
            continue
        candidate_headers = split_row(line)
        separator = split_row(lines[index + 1])
        if len(candidate_headers) == len(separator) and all(
            re.fullmatch(r":?-{3,}:?", part.replace(" ", "")) for part in separator
        ):
            if set(required_columns).issubset(candidate_headers):
                header_index = index
                headers = candidate_headers
                break

    if headers is None:
        expected = " | ".join(required_columns)
        raise ValueError(
            f"Respons {stage_name} tidak memuat tabel Markdown dengan kolom wajib: {expected}. "
            "Salin tabel dari AI dan pastikan nama kolomnya sama dengan prompt."
        )

    rows = []
    for line_number, line in enumerate(lines[header_index + 2:], start=header_index + 3):
        if "|" not in line:
            if rows:
                break
            continue
        values = split_row(line)
        if len(values) != len(headers):
            raise ValueError(
                f"Baris {line_number} pada tabel {stage_name} memiliki {len(values)} kolom; "
                f"seharusnya {len(headers)}. Perbaiki tabel respons AI."
            )
        rows.append(dict(zip(headers, values)))

    if not rows:
        raise ValueError(f"Tabel respons {stage_name} tidak memiliki baris data.")
    result = pd.DataFrame(rows)
    for column in required_columns:
        result[column] = result[column].astype(str).str.strip()
        if result[column].eq("").any():
            raise ValueError(f"Kolom '{column}' pada tabel {stage_name} tidak boleh kosong.")
    return result


### Prompt AI pembanding

In [ ]:
prompt_perbandingan_ai = f"""Bandingkan makna kandidat berdasarkan kutipan ayat yang disediakan.

Kandidat: {kandidat_normal}

Untuk SETIAP ayat, tulis konteks penggunaan, makna menurut Anda, bukti tekstual berupa kutipan dari konteks yang diberikan, dan batas analisis. Pertahankan rujukan persis; jangan menambah klaim di luar teks. Keluarkan hanya tabel Markdown dengan kolom persis: rujukan | konteks_penggunaan | makna_menurut_ai | bukti_tekstual | batasan_analisis.

Konteks ayat lengkap:
{chr(10).join(konteks_ai)}"""
print("===== PROMPT AI PEMBANDING ? salin ke model AI lain =====\n")
print(prompt_perbandingan_ai)

### Tempel hasil AI pembanding (ISI MANUAL)

Salin tabel Markdown lengkap yang dikembalikan AI ke variabel `LLM_AI_PEMBANDING` pada cell kode berikutnya, di antara tanda tiga kutip. Isi penilaian dan catatan Anda setelah membandingkannya dengan kolom ChatGPT awal.

In [ ]:
# Tempel tabel Markdown lengkap dari AI pembanding di antara tanda tiga kutip.
NAMA_AI_PEMBANDING = "Isi nama platform/model AI"
TANGGAL_AI_PEMBANDING = "YYYY-MM-DD"
LLM_AI_PEMBANDING = """| rujukan         | konteks_penggunaan                                                                                                                                                                               | makna_menurut_ai                                                                                                                                                | bukti_tekstual                                                                                                                                                                             | batasan_analisis                                                                                                                                                          |
| --------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ | --------------------------------------------------------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| Al-Baqara 2:8   | Pernyataan tentang keimanan kepada Allah dan Hari kemudian yang kemudian disangkal oleh keterangan ayat.                                                                                         | Orang-orang yang beriman menjadi pembanding bagi orang yang mengaku beriman, tetapi menurut teks sebenarnya bukan bagian dari kelompok tersebut.                | “Kami beriman kepada Allah dan Hari kemudian,” dan “pada hal mereka itu sesungguhnya bukan orang-orang yang beriman.”                                                                      | Makna dibatasi pada pertentangan antara pernyataan dan keterangan dalam ayat. Tidak menyimpulkan keadaan batin individu di luar pernyataan teks.                          |
| Al-Baqara 2:9   | Orang-orang yang beriman disebut sebagai pihak yang hendak ditipu oleh kelompok lain.                                                                                                            | Orang-orang yang beriman berkedudukan sebagai sasaran tindakan menipu dalam interaksi yang dijelaskan ayat.                                                     | “Mereka hendak menipu Allah dan orang-orang yang beriman.”                                                                                                                                 | Ayat menyebut tujuan tindakan kelompok tersebut, tetapi tidak merinci akibat tindakan itu terhadap orang-orang yang beriman.                                              |
| Al-Baqara 2:14  | Perjumpaan dengan orang-orang yang beriman menjadi konteks penyampaian pernyataan keimanan, yang berbeda dari ucapan kepada kelompok lain.                                                       | Orang-orang yang beriman merupakan pihak yang dihadapi ketika kelompok tersebut mengatakan bahwa mereka telah beriman.                                          | “Dan bila mereka berjumpa dengan orang-orang yang beriman, mereka mengatakan: ‘Kami telah beriman’.”                                                                                       | Makna didasarkan pada perbedaan ucapan dalam dua situasi yang disebutkan. Tidak memperluasnya menjadi generalisasi tentang semua interaksi sosial.                        |
| Al-Baqara 2:26  | Orang-orang yang beriman disebut dalam konteks penerimaan perumpamaan yang dibuat Allah, dengan keyakinan bahwa perumpamaan itu benar.                                                           | Orang-orang yang beriman digambarkan sebagai pihak yang meyakini kebenaran perumpamaan tersebut dari Tuhan mereka.                                              | “Adapun orang-orang yang beriman, maka mereka yakin bahwa perumpamaan itu benar dari Tuhan mereka.”                                                                                        | Penjelasan hanya mencakup respons terhadap perumpamaan dalam ayat ini. Tidak menyimpulkan bahwa setiap orang beriman selalu menerima seluruh pesan dengan cara yang sama. |
| Al-Baqara 2:76  | Orang-orang yang beriman menjadi pihak yang ditemui dan diajak berbicara oleh kelompok lain, sementara pembicaraan internal kelompok tersebut membahas informasi yang disampaikan kepada mereka. | Orang-orang yang beriman merupakan pihak yang menerima pernyataan keimanan dan disebut sebagai penerima informasi dalam pembicaraan internal kelompok tersebut. | “Dan apabila mereka berjumpa dengan orang-orang yang beriman, mereka berkata: ‘Kamipun telah beriman’”; “apa yang telah diterangkan Allah kepadamu”; “kepada mereka (orang-orang mukmin)”. | Teks menunjukkan perjumpaan dan kekhawatiran mengenai penyampaian informasi. Tidak cukup untuk menyimpulkan seluruh motif atau tindakan orang-orang yang beriman.         |
| Al-Baqara 2:82  | Orang-orang yang beriman disebut bersama orang-orang yang beramal saleh dalam pernyataan mengenai penghuni surga.                                                                                | Orang-orang yang beriman merupakan bagian dari kelompok yang dikaitkan dengan amal saleh dan konsekuensi berupa surga serta kekekalan di dalamnya.              | “Dan orang-orang yang beriman serta beramal saleh, mereka itu penghuni surga; mereka kekal di dalamnya.”                                                                                   | Makna mengikuti hubungan antara keimanan, amal saleh, dan balasan sebagaimana dinyatakan dalam ayat. Tidak membahas persyaratan lain yang tidak disebutkan dalam kutipan. |
| Al-Baqara 2:91  | Pernyataan tentang keimanan digunakan dalam pertanyaan yang menguji konsistensi antara klaim keimanan dan tindakan terhadap nabi-nabi Allah.                                                     | Orang-orang yang beriman menjadi tolok ukur dalam pertanyaan retoris mengenai kesesuaian antara pengakuan iman dan tindakan yang disebutkan dalam ayat.         | “Katakanlah: ‘Mengapa kamu dahulu membunuh nabi-nabi Allah jika benar kamu orang-orang yang beriman?’”                                                                                     | Pertanyaan tersebut merupakan bagian dari argumen dalam ayat. Analisis tidak menentukan identitas individu tertentu atau memperluas tuduhan di luar teks.                 |
| Al-Baqara 2:97  | Orang-orang yang beriman disebut sebagai penerima petunjuk dan berita gembira yang berkaitan dengan Al-Qur'an.                                                                                   | Orang-orang yang beriman merupakan kelompok yang dituju oleh fungsi petunjuk dan berita gembira dari Al-Qur'an sebagaimana dinyatakan dalam ayat.               | “Membenarkan apa (kitab-kitab) yang sebelumnya dan menjadi petunjuk serta berita gembira bagi orang-orang yang beriman.”                                                                   | Makna terbatas pada fungsi petunjuk dan berita gembira yang dinyatakan dalam kutipan. Tidak menyimpulkan bentuk atau hasil penerimaan setiap individu.                    |
| Al-Baqara 2:104 | Sapaan kepada orang-orang yang beriman mengawali instruksi mengenai kata yang seharusnya digunakan dan perintah untuk mendengarkan.                                                              | Orang-orang yang beriman berkedudukan sebagai penerima langsung instruksi mengenai pilihan ungkapan dan sikap mendengarkan.                                     | “Hai orang-orang yang beriman, janganlah kamu katakan (kepada Muhammad): ‘Raa'ina’, tetapi katakanlah: ‘Unzhurna’, dan ‘dengarlah’.”                                                       | Analisis berfokus pada fungsi sapaan dan isi instruksi. Tidak menyimpulkan alasan linguistik atau historis tambahan di luar keterangan dalam kutipan.                     |

"""
PENILAIAN_AI_PEMBANDING = "belum ditinjau"
CATATAN_AI_PEMBANDING = ""

kolom_ai_pembanding = [
    "rujukan", "konteks_penggunaan", "makna_menurut_ai", "bukti_tekstual", "batasan_analisis"
]
ai_pembanding_table = parse_ai_markdown_table(
    LLM_AI_PEMBANDING, kolom_ai_pembanding, "AI Pembanding"
)

chatgpt_awal_per_ayat = pd.DataFrame([
    {"rujukan": "Al-Baqara 2:8", "konteks_chatgpt_awal": "Pernyataan membedakan pengakuan keimanan dari keadaan sebenarnya.", "makna_chatgpt_awal": "Frasa dipakai sebagai pembanding bagi pihak yang mengaku beriman tetapi disebut bukan orang beriman.", "bukti_chatgpt_awal": "?Kami beriman kepada Allah dan Hari kemudian?; ?sesungguhnya bukan orang-orang yang beriman.?", "batas_chatgpt_awal": "Kriteria terperinci tentang siapa yang benar-benar beriman tidak dijelaskan dalam kutipan."},
    {"rujukan": "Al-Baqara 2:9", "konteks_chatgpt_awal": "Orang beriman menjadi pihak yang hendak ditipu.", "makna_chatgpt_awal": "Frasa menunjuk pihak sasaran upaya penipuan.", "bukti_chatgpt_awal": "?Mereka hendak menipu Allah dan orang-orang yang beriman.?", "batas_chatgpt_awal": "Identitas kelompok dan cara penipuan tidak dijelaskan terperinci."},
    {"rujukan": "Al-Baqara 2:14", "konteks_chatgpt_awal": "Perjumpaan dan percakapan antara kelompok yang berbeda.", "makna_chatgpt_awal": "Orang beriman menjadi pihak yang ditemui; ucapan pihak lain berbeda ketika kembali kepada kelompoknya.", "bukti_chatgpt_awal": "?Bila mereka berjumpa dengan orang-orang yang beriman, mereka mengatakan: Kami telah beriman?; kemudian ?kami hanyalah berolok-olok.?", "batas_chatgpt_awal": "Identitas lengkap kelompok dan latar perjumpaan tidak dipastikan dari kutipan."},
    {"rujukan": "Al-Baqara 2:26", "konteks_chatgpt_awal": "Orang beriman dibandingkan dengan orang kafir dalam menanggapi perumpamaan.", "makna_chatgpt_awal": "Kelompok beriman yakin bahwa perumpamaan itu benar dari Tuhan mereka.", "bukti_chatgpt_awal": "?Adapun orang-orang yang beriman, maka mereka yakin bahwa perumpamaan itu benar dari Tuhan mereka.?", "batas_chatgpt_awal": "Alasan lengkap perbedaan tanggapan tidak dirinci."},
    {"rujukan": "Al-Baqara 2:76", "konteks_chatgpt_awal": "Perjumpaan dan percakapan tentang pengakuan iman serta informasi.", "makna_chatgpt_awal": "Orang beriman menjadi pihak yang ditemui dan disebut dalam kekhawatiran penggunaan informasi untuk mengalahkan hujah.", "bukti_chatgpt_awal": "?Apabila mereka berjumpa dengan orang-orang yang beriman, mereka berkata: Kamipun telah beriman?; ?supaya dengan demikian mereka dapat mengalahkan hujjahmu.?", "batas_chatgpt_awal": "Kutipan tidak cukup untuk menyimpulkan seluruh hubungan sosial atau maksud tiap pihak."},
    {"rujukan": "Al-Baqara 2:82", "konteks_chatgpt_awal": "Frasa disebut bersama amal saleh dan konsekuensi.", "makna_chatgpt_awal": "Orang beriman bersama pelaku amal saleh dikaitkan dengan surga dan kekekalan.", "bukti_chatgpt_awal": "?Orang-orang yang beriman serta beramal saleh, mereka itu penghuni surga; mereka kekal di dalamnya.?", "batas_chatgpt_awal": "Bentuk amal saleh dan kriteria keimanan tidak dirinci."},
    {"rujukan": "Al-Baqara 2:91", "konteks_chatgpt_awal": "Pertanyaan retoris menguji konsistensi pengakuan keimanan.", "makna_chatgpt_awal": "Pengakuan iman dipertanyakan dengan merujuk pada tindakan terhadap nabi-nabi Allah.", "bukti_chatgpt_awal": "?Mengapa kamu dahulu membunuh nabi-nabi Allah jika benar kamu orang-orang yang beriman??", "batas_chatgpt_awal": "Identitas setiap orang dan konteks historis lengkap tidak ditentukan oleh kutipan."},
    {"rujukan": "Al-Baqara 2:97", "konteks_chatgpt_awal": "Penutup pernyataan tentang Al-Qur?an, kitab terdahulu, petunjuk, dan berita gembira.", "makna_chatgpt_awal": "Orang beriman disebut sebagai penerima berita gembira.", "bukti_chatgpt_awal": "?Menjadi petunjuk serta berita gembira bagi orang-orang yang beriman.?", "batas_chatgpt_awal": "Bentuk berita gembira dan karakteristik seluruh penerimanya tidak dirinci."},
    {"rujukan": "Al-Baqara 2:104", "konteks_chatgpt_awal": "Sapaan langsung sebelum larangan dan arahan pilihan kata.", "makna_chatgpt_awal": "Frasa menandai pihak yang dituju oleh instruksi.", "bukti_chatgpt_awal": "?Hai orang-orang yang beriman, janganlah kamu katakan ... tetapi katakanlah ... dan dengarlah.?", "batas_chatgpt_awal": "Alasan terperinci pilihan kata tidak seluruhnya dijelaskan dalam kutipan."},
])

if not ai_pembanding_table.empty:
    if ai_pembanding_table["rujukan"].duplicated().any():
        raise ValueError("Tabel AI pembanding memiliki rujukan duplikat.")
    if set(ai_pembanding_table["rujukan"]) != set(chatgpt_awal_per_ayat["rujukan"]):
        raise ValueError("Tabel AI pembanding harus memuat tepat sembilan rujukan yang sama dengan jawaban ChatGPT awal.")
    perbandingan_ai = chatgpt_awal_per_ayat.merge(
        ai_pembanding_table, on="rujukan", how="inner", validate="one_to_one"
    )
    perbandingan_ai["status_respons_ai_pembanding"] = "Respons AI pembanding sudah ditempel"
else:
    perbandingan_ai = chatgpt_awal_per_ayat.copy()
    for column in kolom_ai_pembanding[1:]:
        perbandingan_ai[column] = ""
    perbandingan_ai["status_respons_ai_pembanding"] = "Belum diisi: tempel tabel jawaban AI pembanding"

perbandingan_ai.insert(1, "ai_pembanding", NAMA_AI_PEMBANDING)
perbandingan_ai.insert(2, "tanggal_ai_pembanding", TANGGAL_AI_PEMBANDING)
# Isi nilai penilaian dan catatan di bagian atas cell, lalu jalankan ulang.
perbandingan_ai["penilaian_anda"] = PENILAIAN_AI_PEMBANDING
perbandingan_ai["catatan"] = CATATAN_AI_PEMBANDING
print("Status:", perbandingan_ai["status_respons_ai_pembanding"].iloc[0])
print("Model:", NAMA_AI_PEMBANDING)
display(perbandingan_ai)


## 8. Tahapan coding sampai hasil akhir

Bagian ini menerapkan alur **Keyword/Phrase ? Open Coding ? Axial Coding ? Selective Coding ? Thematic Coding ? Interpretasi** pada kandidat ?orang-orang yang beriman?. Kandidat muncul pada sembilan ayat dalam cakupan Juz 1 sebanyak 148 ayat. Di setiap tahap, salin prompt ke AI web, tempel respons pada cell ISI MANUAL, lalu tinjau sebelum meneruskannya ke tahap berikutnya. Respons yang ditempel dipakai pada tahap lanjutan. Jika cell respons kosong, notebook menandai penggunaan draf peneliti sebagai fallback; draf tersebut bukan hasil AI. Semua hasil tetap harus ditinjau terhadap ayat sebelum menjadi keputusan penelitian.

### Menetapkan cakupan dan bukti kandidat

Bangun ulang cakupan Al-F?ti?ah 1:1?7 dan Al-Baqarah 2:1?141 dari dataset mentah, validasi 148 ayat, lalu ambil semua kemunculan literal kandidat. Teks terjemahan sumber dipertahankan sebagai bukti.

In [ ]:
import re

KANDIDAT_CODING = "orang-orang yang beriman"
juz1_coding = data.loc[
    ((data["nomor_surat"] == 1) & data["nomor_ayat"].between(1, 7))
    | ((data["nomor_surat"] == 2) & data["nomor_ayat"].between(1, 141))
].copy()
juz1_coding["rujukan"] = juz1_coding.apply(
    lambda row: f"{row['nama_surat']} {row['nomor_surat']}:{row['nomor_ayat']}", axis=1
)
pola_frasa_coding = rf"(?<!\w){re.escape(KANDIDAT_CODING)}(?!\w)"
ayat_frasa_coding = juz1_coding.loc[
    juz1_coding["teks_terjemahan"].str.lower().str.contains(pola_frasa_coding, regex=True)
].copy()

if len(juz1_coding) != 148:
    raise ValueError(f"Cakupan Juz 1 harus 148 ayat; ditemukan {len(juz1_coding)}.")
if len(ayat_frasa_coding) != 9:
    raise ValueError(f"Kandidat diharapkan muncul pada 9 ayat; ditemukan {len(ayat_frasa_coding)}.")

print(f"Cakupan: {len(juz1_coding)} ayat")
print(f"Kandidat: {KANDIDAT_CODING!r}; muncul pada {len(ayat_frasa_coding)} ayat")
display(ayat_frasa_coding[["rujukan", "teks_terjemahan"]].reset_index(drop=True))

Cakupan dan hitungan kandidat diperiksa sebelum coding. Open code di bawah melekat pada rujukan ayat tertentu; satu label tidak dianggap mewakili semua penggunaan frasa.

### Tahap 2 — Open Coding

Open code ditulis sebagai label fenomena yang dekat dengan teks, bukan tema besar. Cell prompt membentuk instruksi dan konteks ayat untuk AI web, mengikuti alur pada notebook contoh. Tabel di bawah merupakan draf analisis berbasis teks dan jawaban AI yang tersedia; bukan respons AI eksternal baru.

In [ ]:
konteks_open_prompt = "\n\n".join(
    f"[{row.rujukan}] {row.teks_terjemahan.strip()}"
    for row in ayat_frasa_coding.itertuples()
)
prompt_open_coding = f"""Anda membantu open coding pada terjemahan Al-Qur'an berbahasa Indonesia.

Kandidat: {KANDIDAT_CODING}
Cakupan: {len(ayat_frasa_coding)} ayat dalam Juz 1.

Tugas:
- Untuk setiap ayat, tulis satu open code singkat dan dekat dengan fenomena dalam teks.
- Sertakan alasan singkat dan rujukan ayat persis.
- Perhatikan negasi, pelaku, sasaran, dan fungsi frasa.
- Jangan melompat ke kategori besar, tema, tafsir, atau klaim yang tidak didukung kutipan.
- Keluarkan tabel Markdown dengan kolom: rujukan | open_code | alasan.

Konteks ayat lengkap:
{konteks_open_prompt}"""
print("===== PROMPT OPEN CODING ? salin ke AI web; tempel hasil pada cell ISI MANUAL berikutnya =====\n")
print(prompt_open_coding)

Prompt ini mengikuti pola notebook contoh: konteks disediakan, tugas dibatasi, dan format keluaran ditetapkan. Draf hasil di bawah tetap ditautkan ke ayat dataset dan tidak diklaim sebagai respons ChatGPT baru.

### Hasil AI ? Open Coding (ISI MANUAL)

Salin prompt Open Coding di atas ke AI web. Tempel **tabel Markdown hasil AI** pada variabel `LLM_STAGE2_OPEN` di bawah di antara tanda tiga kutip yang sudah tersedia. Pertahankan header `rujukan | open_code | alasan`. Tabel inilah yang akan dipakai pada tahap Open Coding; bila belum diisi, notebook memberi label draf peneliti.

Tempel seperti contoh notebook: `LLM_STAGE = """` lalu seluruh tabel respons lalu `"""`.

In [ ]:
MODEL_OPEN_AI = "Isi nama platform/model AI"
LLM_STAGE2_OPEN = """| rujukan         | open_code                                                                                       | alasan                                                                                                                                                                         |
| --------------- | ----------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| Al-Baqara 2:8   | Pengakuan iman disangkal oleh keterangan ayat                                                   | Teks menyandingkan pernyataan “Kami beriman kepada Allah dan Hari kemudian” dengan penegasan bahwa mereka “sesungguhnya bukan orang-orang yang beriman”.                       |
| Al-Baqara 2:9   | Orang beriman menjadi sasaran upaya penipuan                                                    | Frasa “menipu Allah dan orang-orang yang beriman” menyebut orang beriman sebagai sasaran tindakan yang hendak dilakukan.                                                       |
| Al-Baqara 2:14  | Pengakuan iman disampaikan saat berjumpa dengan orang beriman                                   | Ketika berjumpa dengan orang beriman, mereka mengatakan “Kami telah beriman”, tetapi kepada kelompok lain mereka menyatakan “kami hanyalah berolok-olok”.                      |
| Al-Baqara 2:26  | Orang beriman meyakini kebenaran perumpamaan                                                    | Teks secara langsung menyebut bahwa orang-orang yang beriman “yakin bahwa perumpamaan itu benar dari Tuhan mereka”.                                                            |
| Al-Baqara 2:76  | Orang beriman menjadi pihak yang menerima pengakuan iman dan disebut dalam pembicaraan internal | Saat berjumpa dengan orang beriman, mereka berkata “Kamipun telah beriman”. Dalam pembicaraan sesama mereka, orang-orang mukmin disebut sebagai pihak yang menerima informasi. |
| Al-Baqara 2:82  | Keimanan disebut bersama amal saleh dan balasan surga                                           | Teks menghubungkan “orang-orang yang beriman serta beramal saleh” dengan keterangan “penghuni surga” dan “mereka kekal di dalamnya”.                                           |
| Al-Baqara 2:91  | Klaim keimanan dipertanyakan melalui tindakan terhadap nabi-nabi                                | Pertanyaan “Mengapa kamu dahulu membunuh nabi-nabi Allah jika benar kamu orang-orang yang beriman?” mengaitkan klaim keimanan dengan tindakan yang disebutkan.                 |
| Al-Baqara 2:97  | Orang beriman disebut sebagai penerima petunjuk dan berita gembira                              | Frasa “menjadi petunjuk serta berita gembira bagi orang-orang yang beriman” menunjukkan fungsi petunjuk dan berita gembira bagi kelompok tersebut.                             |
| Al-Baqara 2:104 | Orang beriman menjadi sasaran instruksi tentang penggunaan kata                                 | Sapaan “Hai orang-orang yang beriman” diikuti larangan mengatakan “Raa'ina” dan arahan untuk mengatakan “Unzhurna” serta “dengarlah”.                                          |

"""
PENILAIAN_OPEN_AI = "belum ditinjau"
CATATAN_OPEN_AI = "Isi koreksi atau alasan penilaian peneliti"

open_ai_table = parse_ai_markdown_table(
    LLM_STAGE2_OPEN, ["rujukan", "open_code", "alasan"], "Open Coding"
)
print("Model:", MODEL_OPEN_AI)
print("Status:", "respons AI siap dipakai" if not open_ai_table.empty else "belum diisi; tahap ini memakai draf peneliti")
if not open_ai_table.empty:
    open_ai_table["penilaian_anda"] = PENILAIAN_OPEN_AI
    open_ai_table["catatan"] = CATATAN_OPEN_AI
    display(open_ai_table)


Tabel respons yang ditempel akan menjadi input Open Coding setelah rujukan dan kelengkapan baris divalidasi.

In [ ]:
if not open_ai_table.empty:
    if open_ai_table["rujukan"].duplicated().any():
        raise ValueError("Respons Open Coding memiliki rujukan duplikat; pastikan satu baris per ayat.")
    open_code_source = open_ai_table.copy()
    if set(open_code_source["rujukan"]) != set(ayat_frasa_coding["rujukan"]):
        raise ValueError("Rujukan Open Coding harus mencakup tepat sembilan ayat kandidat. Periksa tabel respons AI.")
    open_coding = open_code_source.merge(
        ayat_frasa_coding[["rujukan", "teks_terjemahan"]], on="rujukan", how="left", validate="one_to_one"
    )
    open_coding["asal_draf"] = f"Respons {MODEL_OPEN_AI}; {PENILAIAN_OPEN_AI}"
    open_coding["catatan_peneliti"] = CATATAN_OPEN_AI
else:
    open_code_drafts = [
        {"rujukan": "Al-Baqara 2:8", "open_code": "Penyangkalan terhadap klaim beriman", "alasan": "Kata 'bukan' menyangkal pengakuan 'kami beriman' dalam ayat."},
        {"rujukan": "Al-Baqara 2:9", "open_code": "Orang beriman sebagai sasaran penipuan", "alasan": "Frasa secara langsung menjadi objek dari tindakan hendak menipu."},
        {"rujukan": "Al-Baqara 2:14", "open_code": "Pengakuan berbeda menurut lawan bicara", "alasan": "Ayat membandingkan ucapan saat berjumpa dengan orang beriman dan saat kembali kepada kelompoknya."},
        {"rujukan": "Al-Baqara 2:26", "open_code": "Meyakini kebenaran perumpamaan", "alasan": "Orang beriman disebut yakin bahwa perumpamaan itu benar dari Tuhan mereka."},
        {"rujukan": "Al-Baqara 2:76", "open_code": "Perjumpaan dan kekhawatiran penggunaan informasi", "alasan": "Ayat menyebut perjumpaan dengan orang beriman dan kekhawatiran informasi dipakai untuk mengalahkan hujah."},
        {"rujukan": "Al-Baqara 2:82", "open_code": "Iman dan amal saleh terkait ganjaran", "alasan": "Kelompok disebut bersama pelaku amal saleh dan dikaitkan dengan surga serta kekekalan."},
        {"rujukan": "Al-Baqara 2:91", "open_code": "Klaim iman dipertanyakan melalui tindakan", "alasan": "Pertanyaan menguji klaim beriman dengan merujuk pada tindakan membunuh nabi-nabi Allah."},
        {"rujukan": "Al-Baqara 2:97", "open_code": "Penerima petunjuk dan berita gembira", "alasan": "Frasa menyebut penerima petunjuk serta berita gembira yang dikaitkan dengan Al-Qur'an."},
        {"rujukan": "Al-Baqara 2:104", "open_code": "Penerima sapaan dan arahan ujaran", "alasan": "Frasa menjadi sapaan langsung sebelum larangan dan arahan penggunaan kata."},
    ]
    open_coding = pd.DataFrame(open_code_drafts).merge(
        ayat_frasa_coding[["rujukan", "teks_terjemahan"]], on="rujukan", how="left", validate="one_to_one"
    )
    open_coding["asal_draf"] = "Draf peneliti berbasis teks dataset; bukan respons AI tahap ini"
    open_coding["catatan_peneliti"] = "Tinjau dan koreksi sebelum menetapkan coding"

if open_coding["teks_terjemahan"].isna().any():
    raise ValueError("Ada open code yang rujukannya tidak ditemukan pada ayat kandidat.")
display(open_coding[["rujukan", "teks_terjemahan", "open_code", "alasan", "asal_draf", "catatan_peneliti"]])


Setiap open code terhubung ke satu ayat dan bukti terjemahannya. Istilah pada kolom `asal_draf` menandai bahwa label belum disahkan sebagai keputusan final peneliti.

### Tahap 3 — Axial Coding

Axial coding menghubungkan open code berdasarkan relasi makna yang dapat dijelaskan. Kemiripan leksikal dihitung sebagai bantuan eksploratif saja; kategori berikut tidak ditentukan otomatis oleh skor.

In [ ]:
import math
from collections import Counter

dokumen_kode = [re.findall(r"\w+", code.lower()) for code in open_coding["open_code"]]
frekuensi_dokumen = Counter(token for dokumen in dokumen_kode for token in set(dokumen))
vektor_tfidf = []
for dokumen in dokumen_kode:
    tf = Counter(dokumen)
    vector = {
        token: (1 + math.log(count)) * (math.log((1 + len(dokumen_kode)) / (1 + frekuensi_dokumen[token])) + 1)
        for token, count in tf.items()
    }
    vektor_tfidf.append(vector)

pasangan_kemiripan = []
for i, kode_a in enumerate(vektor_tfidf):
    for j in range(i + 1, len(vektor_tfidf)):
        kode_b = vektor_tfidf[j]
        irisan = set(kode_a) & set(kode_b)
        dot = sum(kode_a[token] * kode_b[token] for token in irisan)
        norm_a = math.sqrt(sum(value * value for value in kode_a.values()))
        norm_b = math.sqrt(sum(value * value for value in kode_b.values()))
        cosine = dot / (norm_a * norm_b) if norm_a and norm_b else 0.0
        pasangan_kemiripan.append({
            "open_code_1": open_coding.iloc[i]["open_code"],
            "open_code_2": open_coding.iloc[j]["open_code"],
            "cosine_tfidf": cosine,
        })
similarity_open_codes = pd.DataFrame(pasangan_kemiripan).sort_values(
    "cosine_tfidf", ascending=False, ignore_index=True
)
display(similarity_open_codes.head(12))

Skor cosine TF-IDF berbasis token ini hanya menunjukkan kemiripan istilah pada sembilan label open code; skor nol tidak berarti maknanya pasti tidak berhubungan, dan skor tinggi bukan bukti kategori yang sama.

In [ ]:
daftar_open_code_prompt = "\n".join(
    f"- {row.rujukan}: {row.open_code} — {row.alasan}"
    for row in open_coding.itertuples()
)
prompt_axial_coding = f"""Anda membantu axial coding pada hasil open coding dari terjemahan Al-Qur'an Juz 1.

Kelompokkan open code yang memiliki hubungan konseptual dan jelaskan alasan hubungan. Jangan mengelompokkan hanya berdasarkan kata yang sama atau skor kemiripan; pertahankan kasus yang berbeda dan jangan menambah tafsir di luar bukti.

Open code:
{daftar_open_code_prompt}

Keluarkan tabel Markdown dengan kolom: rujukan | open_code | axial_category | alasan_hubungan."""
print("===== PROMPT AXIAL CODING ? salin ke AI web; tempel hasil pada cell ISI MANUAL berikutnya =====\n")
print(prompt_axial_coding)

Kategori axial berikut adalah draf yang ditautkan ke kode dan ayat. Prompt menyediakan cara meminta usulan pembanding dari AI web, tetapi tidak mengirimkannya secara otomatis.

### Hasil AI ? Axial Coding (ISI MANUAL)

Salin prompt Axial Coding di atas setelah memeriksa hasil Open Coding. Tempel tabel Markdown pada `LLM_STAGE3_AXIAL` (diapit tanda `"""`) dengan header `rujukan | open_code | axial_category | alasan_hubungan`. Jawaban AI ini akan dipakai untuk membentuk kategori axial; jika kosong, digunakan draf peneliti.

Tempel seperti contoh notebook: `LLM_STAGE = """` lalu seluruh tabel respons lalu `"""`.

In [ ]:
MODEL_AXIAL_AI = "Isi nama platform/model AI"
LLM_STAGE3_AXIAL = """| rujukan         | open_code                                        | axial_category                             | alasan_hubungan                                                                                                                                                                                                                                                                                              |
| --------------- | ------------------------------------------------ | ------------------------------------------ | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| Al-Baqara 2:8   | Penyangkalan terhadap klaim beriman              | Klaim dan pengujian keimanan               | Kata “bukan” menyangkal pengakuan “kami beriman”, sehingga ayat memperlihatkan pertentangan antara pernyataan keimanan dan keterangan teks.                                                                                                                                                                  |
| Al-Baqara 2:91  | Klaim iman dipertanyakan melalui tindakan        | Klaim dan pengujian keimanan               | Pertanyaan tentang tindakan membunuh nabi-nabi Allah digunakan untuk mempertanyakan klaim keimanan. Hubungannya dengan Al-Baqara 2:8 terletak pada pemeriksaan klaim, meskipun bentuknya berbeda: penyangkalan pada ayat 2:8 dan pertanyaan retoris pada ayat 2:91.                                          |
| Al-Baqara 2:9   | Orang beriman sebagai sasaran penipuan           | Relasi sosial dan komunikasi antarkelompok | Orang-orang yang beriman secara eksplisit disebut sebagai sasaran tindakan menipu. Pengelompokan didasarkan pada posisi mereka dalam tindakan yang dilakukan kelompok lain.                                                                                                                                  |
| Al-Baqara 2:14  | Pengakuan berbeda menurut lawan bicara           | Relasi sosial dan komunikasi antarkelompok | Ayat membedakan ucapan berdasarkan pihak yang dihadapi: pengakuan beriman saat berjumpa dengan orang beriman dan pernyataan berolok-olok ketika kembali kepada kelompoknya.                                                                                                                                  |
| Al-Baqara 2:76  | Perjumpaan dan kekhawatiran penggunaan informasi | Relasi sosial dan komunikasi antarkelompok | Ayat memuat perjumpaan dengan orang beriman dan pembicaraan internal mengenai informasi yang dapat digunakan untuk mengalahkan hujah. Hubungannya dengan ayat 2:14 terletak pada konteks perjumpaan dan komunikasi, tetapi persoalan informasi dalam ayat 2:76 tetap menjadi ciri khusus.                    |
| Al-Baqara 2:26  | Meyakini kebenaran perumpamaan                   | Respons terhadap pesan dan petunjuk        | Orang beriman secara eksplisit disebut yakin bahwa perumpamaan itu benar dari Tuhan mereka. Kategori ini berfokus pada respons terhadap perumpamaan, bukan pada klaim keimanan atau interaksi sosial.                                                                                                        |
| Al-Baqara 2:97  | Penerima petunjuk dan berita gembira             | Respons terhadap pesan dan petunjuk        | Orang beriman disebut sebagai penerima petunjuk dan berita gembira yang dikaitkan dengan Al-Qur'an. Hubungannya dengan ayat 2:26 adalah keterkaitan dengan pesan keagamaan, tetapi ayat 2:97 menekankan fungsi petunjuk dan berita gembira, bukan secara langsung pernyataan keyakinan terhadap perumpamaan. |
| Al-Baqara 2:82  | Iman dan amal saleh terkait ganjaran             | Iman, amal, dan konsekuensi                | Keimanan disebut bersama amal saleh dan dikaitkan dengan surga serta kekekalan. Kategori ini mempertahankan hubungan tekstual antara keimanan, tindakan, dan balasan yang tidak dinyatakan dengan pola serupa dalam open code lainnya.                                                                       |
| Al-Baqara 2:104 | Penerima sapaan dan arahan ujaran                | Penerima instruksi                         | Sapaan “Hai orang-orang yang beriman” secara langsung menandai pihak yang dituju oleh larangan dan arahan penggunaan kata. Kategori ini dibedakan dari respons terhadap petunjuk karena fokus utamanya adalah fungsi sapaan dan instruksi, bukan penerimaan atau keyakinan terhadap suatu pesan.             |

"""
PENILAIAN_AXIAL_AI = "belum ditinjau"
CATATAN_AXIAL_AI = "Isi koreksi atau alasan penilaian peneliti"

axial_ai_table = parse_ai_markdown_table(
    LLM_STAGE3_AXIAL,
    ["rujukan", "open_code", "axial_category", "alasan_hubungan"],
    "Axial Coding",
)
print("Model:", MODEL_AXIAL_AI)
print("Status:", "respons AI siap dipakai" if not axial_ai_table.empty else "belum diisi; tahap ini memakai draf peneliti")
if not axial_ai_table.empty:
    axial_ai_table["penilaian_anda"] = PENILAIAN_AXIAL_AI
    axial_ai_table["catatan"] = CATATAN_AXIAL_AI
    display(axial_ai_table)


Respons Axial Coding yang valid akan dipasangkan dengan open code dan diteruskan ke ringkasan kategori.

In [ ]:
if not axial_ai_table.empty:
    if axial_ai_table["rujukan"].duplicated().any():
        raise ValueError("Respons Axial Coding memiliki rujukan duplikat; pastikan satu baris per ayat.")
    if set(axial_ai_table["rujukan"]) != set(open_coding["rujukan"]):
        raise ValueError("Rujukan Axial Coding harus sama dengan rujukan Open Coding.")
    axial_assignments = axial_ai_table.rename(columns={"alasan_hubungan": "hubungan"})
    axial_coding = open_coding.merge(
        axial_assignments[["rujukan", "axial_category", "hubungan", "open_code"]].rename(
            columns={"open_code": "open_code_dari_respons_axial"}
        ), on="rujukan", how="left", validate="one_to_one"
    )
    axial_coding["asal_draf_axial"] = f"Respons {MODEL_AXIAL_AI}; {PENILAIAN_AXIAL_AI}"
    axial_coding["catatan_peneliti_axial"] = CATATAN_AXIAL_AI
else:
    axial_assignments = [
        {"rujukan": "Al-Baqara 2:8", "axial_category": "Klaim dan pengujian keimanan", "hubungan": "Pernyataan negatif menyangkal klaim keimanan."},
        {"rujukan": "Al-Baqara 2:91", "axial_category": "Klaim dan pengujian keimanan", "hubungan": "Klaim keimanan diuji melalui pertanyaan tentang tindakan yang disebut ayat."},
        {"rujukan": "Al-Baqara 2:9", "axial_category": "Relasi sosial dan komunikasi antarkelompok", "hubungan": "Orang beriman menjadi pihak yang dituju dalam tindakan kelompok lain."},
        {"rujukan": "Al-Baqara 2:14", "axial_category": "Relasi sosial dan komunikasi antarkelompok", "hubungan": "Perjumpaan berkaitan dengan perbedaan ucapan kepada kelompok yang berbeda."},
        {"rujukan": "Al-Baqara 2:76", "axial_category": "Relasi sosial dan komunikasi antarkelompok", "hubungan": "Perjumpaan dan pembicaraan internal memuat kekhawatiran tentang penggunaan informasi."},
        {"rujukan": "Al-Baqara 2:26", "axial_category": "Respons terhadap pesan dan petunjuk", "hubungan": "Kelompok beriman menerima perumpamaan sebagai kebenaran dari Tuhan mereka."},
        {"rujukan": "Al-Baqara 2:97", "axial_category": "Respons terhadap pesan dan petunjuk", "hubungan": "Kelompok beriman disebut sebagai penerima petunjuk dan berita gembira."},
        {"rujukan": "Al-Baqara 2:82", "axial_category": "Iman, amal, dan konsekuensi", "hubungan": "Keimanan disebut bersama amal saleh dan konsekuensi surga."},
        {"rujukan": "Al-Baqara 2:104", "axial_category": "Penerima instruksi", "hubungan": "Sapaan menandai pihak yang dituju oleh perintah penggunaan kata."},
    ]
    axial_coding = open_coding.merge(pd.DataFrame(axial_assignments), on="rujukan", how="left", validate="one_to_one")
    axial_coding["asal_draf_axial"] = "Draf peneliti; bukan respons AI tahap ini"
    axial_coding["catatan_peneliti_axial"] = "Tinjau hubungan kategori terhadap bukti"

if axial_coding["axial_category"].isna().any():
    raise ValueError("Ada open code yang belum mendapat kategori axial.")
display(axial_coding[["rujukan", "open_code", "axial_category", "hubungan", "asal_draf_axial"]])

ringkasan_axial = (axial_coding.groupby("axial_category", as_index=False)
                   .agg(jumlah_kode=("open_code", "nunique"),
                        rujukan=("rujukan", lambda values: "; ".join(values))))
display(ringkasan_axial)


Hasil axial mempertahankan lima kelompok relasi. Dua kategori hanya ditopang satu ayat; hal ini ditampilkan apa adanya dan tidak dianggap sebagai pola yang telah kuat.

### Tahap 4 — Selective Coding

Selective coding mencari gagasan yang dapat menghubungkan kategori, sambil mencatat kategori yang tidak sepenuhnya sesuai. Karena hanya satu kandidat frasa yang sedang ditelusuri, core category di bawah dinyatakan **sementara** dan cakupannya dibatasi pada sembilan ayat kandidat.

In [ ]:
kategori_selective_prompt = "\n".join(
    f"- {row.axial_category}: {row.jumlah_kode} open code; rujukan {row.rujukan}"
    for row in ringkasan_axial.itertuples()
)
prompt_selective_coding = f"""Anda membantu selective coding secara terbatas pada kandidat '{KANDIDAT_CODING}' dalam Juz 1.

Kategori axial:
{kategori_selective_prompt}

Usulkan paling banyak satu core category sementara yang dapat menghubungkan kategori-kategori tersebut. Jelaskan hubungan, bukti ayat, kasus menyimpang, dan batas cakupan. Jangan menggeneralisasi ke seluruh Juz 1 dan jangan menetapkan tafsir otoritatif.

Keluarkan satu baris tabel Markdown dengan kolom: core_category | kategori_yang_terhubung | bukti_rujukan | kasus_pembatas | batas_interpretasi."""
print("===== PROMPT SELECTIVE CODING ? salin ke AI web; tempel hasil pada cell ISI MANUAL berikutnya =====\n")
print(prompt_selective_coding)

Prompt selective coding meminta kategori inti sementara dan kasus pembatas. Core category yang ditampilkan tetap draf lokal berbasis tabel axial di atas.

### Hasil AI ? Selective Coding (ISI MANUAL)

Salin prompt Selective Coding di atas. Minta AI mengembalikan **satu baris tabel Markdown** dengan header `core_category | kategori_yang_terhubung | bukti_rujukan | kasus_pembatas | batas_interpretasi`. Tempel hasilnya pada `LLM_STAGE4_SELECTIVE` di antara tanda `"""`. Respons yang sudah ditabelkan akan dipakai sebagai core category; jika belum diisi, digunakan draf peneliti.

Tempel seperti contoh notebook: `LLM_STAGE = """` lalu seluruh tabel respons lalu `"""`.

In [ ]:
MODEL_SELECTIVE_AI = "Isi nama platform/model AI"
LLM_STAGE4_SELECTIVE = """| core_category                                                                                                    | kategori_yang_terhubung                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               | bukti_rujukan                                                                                                                                                                                                                                                                                                                                                                             | kasus_pembatas                                                                                                                                                                                                                                                                               | batas_interpretasi                                                                                                                                                                                                                                                                          |
| ---------------------------------------------------------------------------------------------------------------- | --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| Posisi orang-orang yang beriman dalam klaim, relasi, respons, dan konsekuensi yang dinyatakan dalam konteks ayat | Klaim dan pengujian keimanan: keimanan dinyatakan atau dipertanyakan (Al-Baqara 2:8; 2:91). Relasi sosial dan komunikasi antarkelompok: orang beriman menjadi sasaran penipuan, pihak dalam perjumpaan, dan pihak yang disebut dalam pembicaraan internal (Al-Baqara 2:9; 2:14; 2:76). Respons terhadap pesan dan petunjuk: orang beriman meyakini kebenaran perumpamaan dan disebut sebagai penerima petunjuk serta berita gembira (Al-Baqara 2:26; 2:97). Iman, amal, dan konsekuensi: keimanan disebut bersama amal saleh dan balasan surga (Al-Baqara 2:82). Penerima instruksi: orang beriman menjadi sasaran arahan mengenai penggunaan kata (Al-Baqara 2:104). | Al-Baqara 2:8 dan 2:91 menunjukkan klaim keimanan yang disangkal atau dipertanyakan; 2:9, 2:14, dan 2:76 memperlihatkan posisi orang beriman dalam interaksi sosial; 2:26 dan 2:97 berkaitan dengan kebenaran perumpamaan, petunjuk, dan berita gembira; 2:82 menghubungkan iman dengan amal saleh dan konsekuensi; sedangkan 2:104 menempatkan orang beriman sebagai penerima instruksi. | Al-Baqara 2:104 berfokus pada sapaan dan arahan ujaran, bukan pengujian klaim iman. Al-Baqara 2:82 secara khusus menghubungkan iman, amal saleh, dan balasan. Keduanya menunjukkan bahwa tidak semua kemunculan kandidat dapat dijelaskan hanya melalui relasi sosial atau penerimaan pesan. | Core category ini bersifat sementara dan hanya menghubungkan lima kategori axial berdasarkan sembilan rujukan yang diberikan. Rumusan tersebut merupakan sintesis pola penggunaan frasa, bukan kesimpulan tentang seluruh Juz 1, keadaan batin semua orang beriman, atau tafsir otoritatif. |

"""
PENILAIAN_SELECTIVE_AI = "belum ditinjau"
CATATAN_SELECTIVE_AI = "Isi koreksi atau alasan penilaian peneliti"

selective_ai_table = parse_ai_markdown_table(
    LLM_STAGE4_SELECTIVE,
    ["core_category", "kategori_yang_terhubung", "bukti_rujukan", "kasus_pembatas", "batas_interpretasi"],
    "Selective Coding",
)
if len(selective_ai_table) > 1:
    raise ValueError("Selective Coding mengharapkan satu baris core category; sisakan satu baris yang Anda pilih.")
print("Model:", MODEL_SELECTIVE_AI)
print("Status:", "respons AI siap dipakai" if not selective_ai_table.empty else "belum diisi; tahap ini memakai draf peneliti")
if not selective_ai_table.empty:
    selective_ai_table["penilaian_anda"] = PENILAIAN_SELECTIVE_AI
    selective_ai_table["catatan"] = CATATAN_SELECTIVE_AI
    display(selective_ai_table)


Core category dari respons terstruktur akan diteruskan ke prompt Thematic Coding dan traceability.

In [ ]:
if not selective_ai_table.empty:
    selected = selective_ai_table.iloc[0]
    core_category_draft = {
        "core_category": selected["core_category"],
        "status": f"respons {MODEL_SELECTIVE_AI}; {PENILAIAN_SELECTIVE_AI}",
        "hubungan_kategori": selected["kategori_yang_terhubung"],
        "bukti": selected["bukti_rujukan"],
        "kasus_yang_membatasi": selected["kasus_pembatas"],
        "batas_interpretasi": selected["batas_interpretasi"],
        "catatan_peneliti": CATATAN_SELECTIVE_AI,
    }
else:
    core_category_draft = {
        "core_category": "Posisi orang beriman dibentuk oleh klaim, relasi, dan respons dalam konteks ayat",
        "status": "draf peneliti; belum ada respons AI Selective Coding",
        "hubungan_kategori": (
            "Kategori klaim/pengujian dan relasi sosial menunjukkan bagaimana kelompok disebut dalam interaksi; "
            "respons terhadap pesan, amal/konsekuensi, dan instruksi menunjukkan fungsi lain frasa pada ayat yang berbeda."
        ),
        "bukti": "Al-Baqara 2:8, 2:9, 2:14, 2:26, 2:76, 2:82, 2:91, 2:97, 2:104",
        "kasus_yang_membatasi": (
            "Al-Baqara 2:8 memuat penyangkalan ('bukan orang-orang yang beriman'), bukan penyebutan positif; "
            "2:104 adalah sapaan instruksional, sehingga tidak dapat disamakan dengan kategori klaim keimanan."
        ),
        "batas_interpretasi": "Hanya mencakup sembilan ayat kandidat dan tidak mewakili seluruh Juz 1.",
        "catatan_peneliti": "Tinjau core category terhadap semua kategori dan bukti ayat.",
    }
display(pd.DataFrame([core_category_draft]))
print("Keputusan metodologis: core category hanya mencakup kandidat dan ayat yang diperiksa; peneliti tetap memvalidasi respons AI atau draf.")


Core category menghubungkan variasi fungsi frasa sebagai sintesis kerja, tetapi kasus penyangkalan dan sapaan langsung membatasi keseragamannya. Jangan menggeneralisasi rumusan ini ke seluruh Juz 1 tanpa meninjau kandidat dan ayat lain.

### Tahap 5 — Thematic Coding dan interpretasi

Tema dirumuskan dari kategori dan bukti yang ditautkan. Tema di bawah tetap spesifik per kelompok ayat; interpretasi sintesis menyebutkan batasnya agar tidak berubah menjadi klaim tafsir otoritatif.

In [ ]:
kategori_tema_prompt = "\n".join(
    f"- {row.axial_category}: {row.hubungan} (rujukan: {row.rujukan})"
    for row in axial_coding.itertuples()
)
prompt_thematic_coding = f"""Anda membantu thematic coding untuk analisis eksploratif terjemahan Al-Qur'an Juz 1.

Kandidat: {KANDIDAT_CODING}
Core category sementara: {core_category_draft['core_category']}

Kategori dan bukti:
{kategori_tema_prompt}

Rumuskan tema yang tetap dekat dengan bukti. Pertahankan tema berbeda bila konteksnya berbeda. Sertakan rujukan pendukung, kasus negatif/pembatas, dan batas interpretasi. Jangan mengklaim mewakili seluruh Juz 1.

Keluarkan tabel Markdown dengan kolom persis: axial_category | tema | rujukan_tema | batas_interpretasi."""
print("===== PROMPT THEMATIC CODING ? salin ke AI web; tempel hasil pada cell ISI MANUAL berikutnya =====\n")
print(prompt_thematic_coding)

Tema dan interpretasi berikut merupakan draf sintesis berbasis kode dan kutipan. Tidak ada respons AI eksternal baru yang diklaim pada tahap ini.

### Hasil AI ? Thematic Coding (ISI MANUAL)

Salin prompt Thematic Coding di atas. Tempel tabel Markdown pada `LLM_STAGE5_THEMATIC` (diapit tanda `"""`) dengan header `axial_category | tema | rujukan_tema | batas_interpretasi`. Jawaban AI ini akan dipakai sebagai tabel tema. Pastikan semua kategori axial terwakili; jika belum diisi, notebook menggunakan draf peneliti.

Tempel seperti contoh notebook: `LLM_STAGE = """` lalu seluruh tabel respons lalu `"""`.

In [ ]:
MODEL_THEMATIC_AI = "Isi nama platform/model AI"
LLM_STAGE5_THEMATIC = """| axial_category                             | tema                                                                                                                                                                                                                                                                                         | rujukan_tema              | batas_interpretasi                                                                                                                                                                                                                                                                                                              |
| ------------------------------------------ | -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| Klaim dan pengujian keimanan               | Klaim keimanan dapat disangkal atau dipertanyakan melalui pernyataan dan tindakan yang disebutkan dalam ayat. Pada satu konteks, pengakuan beriman disangkal secara eksplisit; pada konteks lain, klaim tersebut dipertanyakan melalui pertanyaan tentang tindakan terhadap nabi-nabi Allah. | Al-Baqara 2:8, 2:91       | Kedua ayat memiliki bentuk berbeda: Al-Baqara 2:8 menyatakan penyangkalan, sedangkan 2:91 memuat pertanyaan retoris. Keduanya tidak cukup untuk menyimpulkan keadaan batin individu di luar teks.                                                                                                                               |
| Relasi sosial dan komunikasi antarkelompok | Orang-orang yang beriman ditempatkan sebagai pihak dalam interaksi yang melibatkan penipuan, perbedaan ucapan, dan pembicaraan tentang informasi.                                                                                                                                            | Al-Baqara 2:9, 2:14, 2:76 | Ketiga ayat memiliki fokus yang berbeda. Al-Baqara 2:9 menyebut orang beriman sebagai sasaran penipuan; 2:14 memperlihatkan perbedaan ucapan menurut lawan bicara; dan 2:76 memuat kekhawatiran tentang penyampaian informasi. Tema ini tidak membuktikan bahwa seluruh interaksi antarkelompok berlangsung dengan pola serupa. |
| Respons terhadap pesan dan petunjuk        | Orang-orang yang beriman dikaitkan dengan penerimaan kebenaran perumpamaan serta fungsi petunjuk dan berita gembira.                                                                                                                                                                         | Al-Baqara 2:26, 2:97      | Al-Baqara 2:26 secara eksplisit menyebut keyakinan terhadap kebenaran perumpamaan, sedangkan 2:97 menyebut petunjuk dan berita gembira bagi orang beriman. Keduanya berkaitan dengan pesan keagamaan, tetapi tidak menunjukkan respons yang identik.                                                                            |
| Iman, amal, dan konsekuensi                | Keimanan disebut bersama amal saleh dalam kaitannya dengan surga dan kekekalan di dalamnya.                                                                                                                                                                                                  | Al-Baqara 2:82            | Tema ini hanya didukung oleh satu rujukan dalam kumpulan data. Tidak tepat menggeneralisasikannya menjadi penjelasan lengkap tentang seluruh hubungan iman, amal, dan balasan dalam Juz 1.                                                                                                                                      |
| Penerima instruksi                         | Sapaan kepada orang-orang yang beriman menandai sasaran instruksi mengenai penggunaan kata dan tindakan mendengarkan.                                                                                                                                                                        | Al-Baqara 2:104           | Ayat ini berfokus pada arahan langsung, bukan pada penyangkalan klaim keimanan, interaksi penipuan, atau penerimaan perumpamaan. Tidak ada bukti dalam kutipan bahwa instruksi tersebut selalu dipatuhi.                                                                                                                        |

"""
PENILAIAN_THEMATIC_AI = "belum ditinjau"
CATATAN_THEMATIC_AI = "Isi koreksi atau alasan penilaian peneliti"

thematic_ai_table = parse_ai_markdown_table(
    LLM_STAGE5_THEMATIC,
    ["axial_category", "tema", "rujukan_tema", "batas_interpretasi"],
    "Thematic Coding",
)
print("Model:", MODEL_THEMATIC_AI)
print("Status:", "respons AI siap dipakai" if not thematic_ai_table.empty else "belum diisi; tahap ini memakai draf peneliti")
if not thematic_ai_table.empty:
    thematic_ai_table["penilaian_anda"] = PENILAIAN_THEMATIC_AI
    thematic_ai_table["catatan"] = CATATAN_THEMATIC_AI
    display(thematic_ai_table)


Tema dari respons terstruktur akan dipakai dalam traceability dan ekspor hasil.

In [ ]:
if not thematic_ai_table.empty:
    if thematic_ai_table["axial_category"].duplicated().any():
        raise ValueError("Respons Thematic Coding memiliki kategori axial duplikat.")
    expected_categories = set(axial_coding["axial_category"])
    if set(thematic_ai_table["axial_category"]) != expected_categories:
        raise ValueError("Tabel Thematic Coding harus mencakup tepat semua kategori axial.")
    thematic_coding = thematic_ai_table.copy()
    thematic_coding["status_tema"] = f"Respons {MODEL_THEMATIC_AI}; {PENILAIAN_THEMATIC_AI}"
    thematic_coding["catatan_peneliti"] = CATATAN_THEMATIC_AI
    interpretasi_sintesis = "Interpretasi sintesis perlu dirumuskan dan ditinjau peneliti berdasarkan tema AI serta teks ayat lengkap."
    batas_interpretasi = "; ".join(thematic_coding["batas_interpretasi"].drop_duplicates())
else:
    thematic_assignments = [
        {"axial_category": "Klaim dan pengujian keimanan", "tema": "Klaim keimanan dinilai dalam hubungannya dengan penyangkalan dan tindakan yang disebut ayat.", "rujukan_tema": "Al-Baqara 2:8; 2:91", "batas_interpretasi": "Terbatas pada konteks klaim dan tindakan dalam dua ayat."},
        {"axial_category": "Relasi sosial dan komunikasi antarkelompok", "tema": "Orang beriman hadir sebagai pihak dalam perjumpaan, komunikasi, dan tindakan sosial antarkelompok.", "rujukan_tema": "Al-Baqara 2:9; 2:14; 2:76", "batas_interpretasi": "Tidak merinci seluruh identitas atau latar hubungan kelompok."},
        {"axial_category": "Respons terhadap pesan dan petunjuk", "tema": "Orang beriman digambarkan menerima kebenaran perumpamaan serta menjadi penerima petunjuk dan berita gembira.", "rujukan_tema": "Al-Baqara 2:26; 2:97", "batas_interpretasi": "Terbatas pada respons dan berita yang disebut dalam ayat."},
        {"axial_category": "Iman, amal, dan konsekuensi", "tema": "Keimanan disebut bersama amal saleh dan konsekuensi yang dinyatakan dalam ayat.", "rujukan_tema": "Al-Baqara 2:82", "batas_interpretasi": "Ayat ini tidak merinci seluruh bentuk amal saleh."},
        {"axial_category": "Penerima instruksi", "tema": "Frasa berfungsi sebagai sapaan kepada pihak yang menerima arahan penggunaan kata.", "rujukan_tema": "Al-Baqara 2:104", "batas_interpretasi": "Fungsi sapaan ini berbeda dari uraian deskriptif tentang kelompok."},
    ]
    thematic_coding = pd.DataFrame(thematic_assignments)
    thematic_coding["status_tema"] = "Draf peneliti; bukan respons AI Thematic Coding"
    thematic_coding["catatan_peneliti"] = "Tinjau tema terhadap axial category dan ayat"
    interpretasi_sintesis = (
        "Pada sembilan ayat yang memuat frasa 'orang-orang yang beriman', frasa tidak menjalankan satu fungsi tunggal: "
        "ia muncul dalam penyangkalan dan pengujian klaim, relasi sosial, penerimaan pesan, kaitan iman dengan amal dan konsekuensi, "
        "serta sapaan instruksional. Karena itu, tema yang paling aman adalah variasi posisi frasa sesuai konteks ayat, bukan satu sifat "
        "universal seluruh orang beriman. Temuan ini bersifat eksploratif dan terbatas pada kandidat serta ayat Juz 1 yang diperiksa."
    )
    batas_interpretasi = (
        "Analisis menggunakan satu terjemahan Indonesia dan satu frasa literal; tidak menentukan identitas kelompok di luar teks, "
        "tidak menyimpulkan tafsir historis/teologis, dan tidak menganggap frekuensi sebagai ukuran kepentingan makna."
    )
display(thematic_coding)
print("Interpretasi sintesis:\n", interpretasi_sintesis)
print("\nBatas interpretasi:\n", batas_interpretasi)


Tema dan interpretasi mengacu pada sembilan rujukan kandidat. Al-Baqara 2:8 dipertahankan sebagai kasus penting yang membatasi pembacaan positif seragam terhadap frasa.

### Evaluasi AI

Evaluasi berikut membandingkan jawaban ChatGPT yang diberikan pengguna dengan teks ayat lengkap di dataset. Kekuatan, kekurangan, dan batas penggunaan AI dicatat tanpa menganggap AI sebagai penentu hasil coding.

In [ ]:
evaluasi_ai = pd.DataFrame([
    {"aspek": "Kontekstualisasi", "temuan": "AI membedakan penggunaan frasa dalam interaksi antarkelompok, tanggapan terhadap perumpamaan, amal saleh, petunjuk, dan instruksi.", "penilaian": "Kekuatan; membantu menghindari penyamaan semua kemunculan."},
    {"aspek": "Dukungan bukti", "temuan": "Sebagian besar rujukan dan kutipan yang disebut cocok dengan dataset; potongan ayat tetap perlu dibandingkan dengan terjemahan lengkap.", "penilaian": "Cukup baik, tetapi pemeriksaan sumber tetap diperlukan."},
    {"aspek": "Negasi pada 2:8", "temuan": "Frasa muncul dalam pernyataan 'bukan orang-orang yang beriman'; uraian AI berisiko membacanya sebagai sebutan positif bagi kelompok beriman.", "penilaian": "Perlu koreksi; negasi mengubah arah klausa."},
    {"aspek": "Kelengkapan konteks", "temuan": "Pada 2:9 AI tidak menonjolkan kelanjutan bahwa mereka menipu diri sendiri; pada 2:97 petunjuk disebut bersama berita gembira.", "penilaian": "Perlu dilengkapi dengan membaca ayat utuh."},
    {"aspek": "Batas interpretasi", "temuan": "AI kerap memberi batasan atas informasi yang tidak dapat dipastikan hanya dari kutipan.", "penilaian": "Kekuatan; selaras dengan pembacaan yang berhati-hati."},
    {"aspek": "Keputusan final", "temuan": "Open code, kategori, tema, dan interpretasi disusun sebagai draf kerja dari bukti; bukan keluaran AI eksternal baru.", "penilaian": "Wajib ditinjau dan disahkan peneliti."},
])
display(evaluasi_ai)

kandidat_export = pd.DataFrame([{
    "jenis_kandidat": "phrase",
    "kandidat": KANDIDAT_CODING,
    "frekuensi_literal": sum(
        len(re.findall(pola_frasa_coding, text.lower()))
        for text in ayat_frasa_coding["teks_terjemahan"]
    ),
    "jumlah_ayat": len(ayat_frasa_coding),
    "rujukan": "; ".join(ayat_frasa_coding["rujukan"]),
}])
log_ai_awal = pd.DataFrame([{
    "platform": "ChatGPT web",
    "tanggal_dicatat": "2026-10-22",
    "kandidat": KANDIDAT_CODING,
    "cakupan_konteks": "9 ayat; Al-Baqara 2:8, 2:9, 2:14, 2:26, 2:76, 2:82, 2:91, 2:97, 2:104",
    "status_respons": "Respons awal diberikan pengguna; diperiksa terhadap dataset",
    "catatan": "Notebook tidak mengirim permintaan API; log ini mencatat respons web yang diberikan pengguna.",
}])
display(kandidat_export)
display(log_ai_awal)

## Evaluasi Anda sebagai pengguna AI

Seperti notebook contoh, nilai hasil AI pada setiap tahap. Ulangi prompt dengan model lain bila ingin membandingkan konsistensi. Isi kolom kelebihan, kekurangan, konteks, koreksi, dan konsistensi secara manual setelah memeriksa bukti ayat. Status draf bukan penilaian terhadap AI.

In [ ]:
evaluasi_proses_ai = pd.DataFrame([
    {"tahap": "1 Keyword/Phrase", "llm_yang_dipakai_dan_versi": "ChatGPT web (respons yang diberikan pengguna); pembanding: " + NAMA_AI_PEMBANDING, "kelebihan": "ISI MANUAL", "kekurangan": "ISI MANUAL", "terlalu_umum_atau_melompat_ke_tema": "ISI MANUAL", "paham_konteks": "ISI MANUAL", "perlu_koreksi_manual": "ISI MANUAL", "konsistensi_saat_diulang_atau_dibandingkan": "ISI MANUAL"},
    {"tahap": "2 Open Coding", "llm_yang_dipakai_dan_versi": MODEL_OPEN_AI if not open_ai_table.empty else "Belum ada respons AI; memakai draf peneliti", "kelebihan": "ISI MANUAL", "kekurangan": "ISI MANUAL", "terlalu_umum_atau_melompat_ke_tema": "ISI MANUAL", "paham_konteks": "ISI MANUAL", "perlu_koreksi_manual": "ISI MANUAL", "konsistensi_saat_diulang_atau_dibandingkan": "ISI MANUAL"},
    {"tahap": "3 Axial Coding", "llm_yang_dipakai_dan_versi": MODEL_AXIAL_AI if not axial_ai_table.empty else "Belum ada respons AI; memakai draf peneliti", "kelebihan": "ISI MANUAL", "kekurangan": "ISI MANUAL", "terlalu_umum_atau_melompat_ke_tema": "ISI MANUAL", "paham_konteks": "ISI MANUAL", "perlu_koreksi_manual": "ISI MANUAL", "konsistensi_saat_diulang_atau_dibandingkan": "ISI MANUAL"},
    {"tahap": "4 Selective Coding", "llm_yang_dipakai_dan_versi": MODEL_SELECTIVE_AI if not selective_ai_table.empty else "Belum ada respons AI; memakai draf peneliti", "kelebihan": "ISI MANUAL", "kekurangan": "ISI MANUAL", "terlalu_umum_atau_melompat_ke_tema": "ISI MANUAL", "paham_konteks": "ISI MANUAL", "perlu_koreksi_manual": "ISI MANUAL", "konsistensi_saat_diulang_atau_dibandingkan": "ISI MANUAL"},
    {"tahap": "5 Thematic Coding", "llm_yang_dipakai_dan_versi": MODEL_THEMATIC_AI if not thematic_ai_table.empty else "Belum ada respons AI; memakai draf peneliti", "kelebihan": "ISI MANUAL", "kekurangan": "ISI MANUAL", "terlalu_umum_atau_melompat_ke_tema": "ISI MANUAL", "paham_konteks": "ISI MANUAL", "perlu_koreksi_manual": "ISI MANUAL", "konsistensi_saat_diulang_atau_dibandingkan": "ISI MANUAL"},
])
display(evaluasi_proses_ai)


Log AI membedakan respons awal ChatGPT yang benar-benar diberikan pengguna dari evaluasi dan coding draf yang dibuat di notebook.

### Traceability dan ekspor hasil

Tabel traceability merangkai bukti ayat → open code → axial category → tema → core category. File CSV disimpan pada `data/outputs/` agar hasil dapat diperiksa di luar notebook.

In [ ]:
traceability_coding = (axial_coding
    .merge(thematic_coding, on="axial_category", how="left", validate="many_to_one")
    .assign(core_category=core_category_draft["core_category"],
            interpretasi_sintesis=interpretasi_sintesis,
            batas_interpretasi=batas_interpretasi,
            status_review="draf; perlu validasi peneliti"))
if traceability_coding["tema"].isna().any():
    raise ValueError("Ada kategori axial yang belum dipetakan ke tema.")

display(traceability_coding[[
    "rujukan", "teks_terjemahan", "open_code", "axial_category", "hubungan", "tema", "core_category", "status_review"
]])

output_dir = Path("data") / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)
kolom_kandidat = ["jenis", "kandidat", "frekuensi", "jumlah_ayat", "rujukan_ayat", "contoh_kutipan"]
keyword_export = kandidat_keyword_awal.rename(columns={"frekuensi_kata": "frekuensi"}).reindex(columns=kolom_kandidat)
phrase_export = kandidat_phrase_awal.rename(columns={"phrase": "kandidat"}).reindex(columns=kolom_kandidat)
keyword_phrase_export = pd.concat([keyword_export, phrase_export], ignore_index=True)
hasil_coding_export = traceability_coding.copy()
hasil_coding_export.to_csv(output_dir / "hasil_coding_juz1.csv", index=False, encoding="utf-8-sig")
keyword_phrase_export.to_csv(output_dir / "keyword_phrase_juz1.csv", index=False, encoding="utf-8-sig")
open_coding.to_csv(output_dir / "open_coding_juz1.csv", index=False, encoding="utf-8-sig")
axial_coding.to_csv(output_dir / "axial_coding_juz1.csv", index=False, encoding="utf-8-sig")
thematic_coding.to_csv(output_dir / "thematic_coding_juz1.csv", index=False, encoding="utf-8-sig")
traceability_coding.to_csv(output_dir / "traceability_coding_juz1.csv", index=False, encoding="utf-8-sig")
pd.DataFrame([core_category_draft]).to_csv(output_dir / "selective_coding_juz1.csv", index=False, encoding="utf-8-sig")
kandidat_export.to_csv(output_dir / "kandidat_coding_juz1.csv", index=False, encoding="utf-8-sig")
log_ai_awal.to_csv(output_dir / "log_ai_juz1.csv", index=False, encoding="utf-8-sig")
evaluasi_ai.to_csv(output_dir / "evaluasi_ai_juz1.csv", index=False, encoding="utf-8-sig")
perbandingan_ai.to_csv(output_dir / "perbandingan_ai_juz1.csv", index=False, encoding="utf-8-sig")
evaluasi_proses_ai.to_csv(output_dir / "evaluasi_proses_ai_juz1.csv", index=False, encoding="utf-8-sig")
print(f"Tabel kandidat, coding lengkap, log AI, evaluasi AI per tahap, perbandingan AI, dan tabel per tahap diekspor ke: {output_dir.resolve()}")

Ekspor menghasilkan tabel kandidat, log/evaluasi AI, Open, Axial, Selective, Thematic, dan Traceability. Setiap baris coding mempertahankan rujukan dan teks ayat agar interpretasi dapat diperiksa kembali.